In [1]:
import numpy as np
import pdb

import control
import matplotlib.pyplot as plt
import torch
import torchsde
from torch import nn, profiler, Tensor, nan, vmap
import copy
import matplotlib.pyplot as plt

from tqdm import tqdm
import numpy as np
import pdb

import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..')) 
sys.path.append(project_root)  # Add folder_a to sys.path
sys.path.append(os.path.join(project_root, '../..', 'Networks'))  # Add folder_b to sys.path

from Networks.utils import get_gpu_memory, normal, ema_update,mass_append, update_ema, generate_system_canon, load_clean_state_dict, sample_linear, compute_gradnorm,save_model, loss_compute
from helpers import   HyperCoeffsLinearControlStochasticLQRImpl,  find_params
import psutil

from IPython.display import clear_output
from adabelief_pytorch import AdaBelief

is_cuda = torch.cuda.is_available()
device = 'cuda' if is_cuda else 'cpu'

if not is_cuda:
    print("Warning: CUDA not available; falling back to CPU but this is likely to be very slow.")
import gc
path=os.path.abspath(os.getcwd())
gc.collect()
torch.cuda.empty_cache()

In [3]:
state_size=128

batch_size=8*256

weight=0.1

A,B=generate_system_canon(state_size)



In [13]:
gc.collect()
torch.cuda.empty_cache()            

global_means=[]
global_stds=[]
At=torch.tensor(A).float().to(device) 
Bt=torch.tensor(B).float().to(device) 
coremat=(At.unsqueeze(0)).repeat(batch_size,1,1)

t_total=1
t_size=int(32)
ts_exp = torch.linspace(0, t_total, t_size).float().to(device)


for layers in range(1,2):
   
    
    strategy="HNC"
    SMLt= HyperCoeffsLinearControlStochasticLQRImpl(At,Bt,batch_size, device,layers,strategy).to(device) 
    load_clean_state_dict(SMLt,  path+"/" + strategy + "ema" + str(layers)+ ".pth")
    
    kinarr=[]
    objarr=[]
    stats=[]
    seed=0
    torch.manual_seed(seed)

    count=0
    mean=0
    M2=0
    for j in tqdm(range(0,2500)):

        amat, x0= sample_linear(dim,batch_size, device)   
        
        ini_enr = torch.zeros(batch_size, 1).to(device)
        ys_exp = torch.cat((x0,ini_enr),dim=1).to(device)
        
        coremat[:,-1,:]=amat
        SMLt.At= coremat
        SMLt.poly.amat=amat

        with torch.no_grad():
            ys_tray = torchsde.sdeint_adjoint(SMLt, ys_exp, ts_exp,method='reversible_heun', dt=ts_exp[1]-ts_exp[0],
                                 adjoint_method='adjoint_reversible_heun',)
            

        print(psutil.virtual_memory())
        dif=torch.sum(ys_tray[-1,:,0:dim]**2,dim=1)
        dif1=weight*ys_tray[-1,:,-1]   
        diff=dif+dif1

        batch=diff.detach().cpu().numpy()
        b_count=batch.size
        b_mean=batch.mean()
        b_M2 = ((batch - b_mean) ** 2).sum()
        delta = b_mean - mean
        total = count + b_count    
        mean += delta * b_count / total
        M2 += b_M2 + delta**2 * count * b_count / total    
        count = total
       
        print("current ste")
        print(np.sqrt(M2/ (count-1))/np.sqrt(count))
        print("current mean")
        print(mean)
        gc.collect()
        torch.cuda.empty_cache()
        del ys_tray,x0
    variance = M2/ (count-1)
    ste= np.sqrt(variance)/np.sqrt(count)
   
    global_means.append(mean)
    global_stds.append(ste)

  0%|          | 1/2500 [00:00<05:32,  7.52it/s]

svmem(total=67025797120, available=47288406016, percent=29.4, used=18355908608, free=35639369728, active=19519033344, inactive=9351958528, buffers=1380290560, cached=11650228224, shared=634990592, slab=1145294848)
current ste
0.008921323542006853
current mean
2.407238483428955
svmem(total=67025797120, available=47288659968, percent=29.4, used=18355388416, free=35639631872, active=19520294912, inactive=9351950336, buffers=1380294656, cached=11650482176, shared=635252736, slab=1145294848)
current ste
0.006159568290412135
current mean
2.4073286056518555


  0%|          | 3/2500 [00:00<05:50,  7.13it/s]

svmem(total=67025797120, available=47288401920, percent=29.4, used=18355646464, free=35639373824, active=19520794624, inactive=9351950336, buffers=1380294656, cached=11650482176, shared=635252736, slab=1145294848)
current ste
0.005016953860973722
current mean
2.4060219128926597
svmem(total=67025797120, available=47279415296, percent=29.5, used=18364633088, free=35630387200, active=19520970752, inactive=9351950336, buffers=1380294656, cached=11650482176, shared=635252736, slab=1145430016)
current ste
0.00432838326065863
current mean
2.4072495698928833


  0%|          | 5/2500 [00:00<05:57,  6.98it/s]

...

 75%|███████▌  | 1887/2500 [04:21<01:25,  7.13it/s]

svmem(total=67025797120, available=48532934656, percent=27.6, used=17118523392, free=36971057152, active=18187038720, inactive=9264988160, buffers=1380532224, cached=11555684352, shared=627863552, slab=1146028032)
current ste
0.00019870101663270982
current mean
2.405434357785143
svmem(total=67025797120, available=48531902464, percent=27.6, used=17120051200, free=36970024960, active=18186850304, inactive=9264988160, buffers=1380532224, cached=11555188736, shared=627367936, slab=1146028032)
current ste
0.00019865199537541202
current mean
2.4054421361977774


Exception ignored in: <function _xla_gc_callback at 0x7c158a9f9080>
Traceback (most recent call last):
  File "/home/qtomcat/anaconda3/lib/python3.12/site-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 
 76%|███████▌  | 1889/2500 [04:21<01:24,  7.21it/s]


svmem(total=67025797120, available=48532193280, percent=27.6, used=17119764480, free=36970311680, active=18187165696, inactive=9264992256, buffers=1380532224, cached=11555188736, shared=627367936, slab=1146028032)
current ste
0.00019859927018383273
current mean
2.4054510500015818



KeyboardInterrupt

